# 02_test_fl_5_clients.ipynb
## Particionado federado y medición por cliente

Este notebook tiene como objetivo validar un primer escenario de Aprendizaje Federado (FL) mediante la partición del conjunto de datos CIFAR-10 en cinco clientes independientes. Cada cliente entrena localmente una instancia del mismo modelo utilizando únicamente su subconjunto de datos, simulando el comportamiento básico de un entorno federado.

Además del entrenamiento distribuido, se integra CodeCarbon para obtener métricas energéticas y de emisiones de forma independiente para cada cliente. Como resultado, este notebook permite verificar que la instrumentación energética funciona correctamente en un contexto federado y constituye el primer paso hacia la construcción del dataset experimental utilizado posteriormente para el análisis de sostenibilidad y el cálculo de SCI.

Las principales validaciones realizadas en esta fase son:

- División del dataset en cinco particiones equivalentes.
- Entrenamiento local independiente por cliente.
- Captura individualizada de energía consumida, emisiones y duración.
- Generación de registros experimentales estructurados para su almacenamiento en el dataset.
- Exportación automática de resultados a formato CSV para su posterior análisis.

In [1]:
DATA_DIR = "../data"

DATASET = "CIFAR10"

SCENARIO = "federated"
NUM_CLIENTS = 5

BATCH_SIZE = 64

EPOCHS = 5

LEARNING_RATE = 0.001

RANDOM_SEED = 42


1. Carga del Dataset y DataLoader

In [2]:

import torch

import torch.nn as nn

from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import datasets
from torchvision import transforms

transform = transforms.ToTensor()

# ==================================================
# Dataset
# ==================================================

train_dataset = datasets.CIFAR10(
    root="../data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

# ==================================================
# Partición FL-5
# ==================================================

client_size = len(train_dataset) // NUM_CLIENTS

client_datasets = random_split(
    train_dataset,
    [client_size] * NUM_CLIENTS
)

# ==================================================
# DataLoaders de clientes
# ==================================================

client_loaders = [
    DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    for ds in client_datasets
]

# ==================================================
# Test loader global
# ==================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


2. Modelo

In [3]:
import torch.nn as nn
import torch.optim as optim

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [5]:
model = SimpleCNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

3. Función de pérdida y optimización

In [ ]:
import torch.optim as optim
import torch.nn as nn
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

4. Función de entrenamiento

In [4]:
def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in train_loader:

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {epoch_loss:.4f}"
        )

5. Función de evaluación

In [ ]:
def evaluate_model(
    model,
    test_loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = correct / total

    return accuracy

6. Función principal

arranca CodeCarbon,
entrena,
evalúa,
crea la fila del dataset,
la guarda en CSV.

In [ ]:
from codecarbon import EmissionsTracker

tracker = EmissionsTracker(
    save_to_file=False
)

tracker.start()

tracker.start_task("client_1")

train_model(
    model,
    client_loaders[0],
    optimizer,
    criterion,
    EPOCHS
)

client_data = tracker.stop_task()

tracker.stop()

print("Energy:", client_data.energy_consumed)
print("Emissions:", client_data.emissions)
print("Duration:", client_data.duration)

[codecarbon WARNING @ 10:07:10] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 10:07:10] [setup] RAM Tracking...
[codecarbon INFO @ 10:07:10] [setup] CPU Tracking...
[codecarbon INFO @ 10:07:10] [setup] GPU Tracking...
[codecarbon INFO @ 10:07:10] No GPU found.
[codecarbon WARNING @ 10:07:12] We saw that you have a 12th Gen Intel(R) Core(TM) i7-1255U but we don't know it. Please contact us.
[codecarbon WARNING @ 10:07:12] We will use the default power consumption of 4 W per thread for your 12 CPU, so 48W.
[codecarbon WARNING @ 10:07:12] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 10:07:12] CPU Model on constant consumption mode: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon WARNING @ 10:07:12] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon

Epoch 1/5 - Loss: 1.8605
Epoch 2/5 - Loss: 1.4694
Epoch 3/5 - Loss: 1.3288
Epoch 4/5 - Loss: 1.2352


[codecarbon INFO @ 10:07:49] Energy consumed for RAM : 0.000095 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 10:07:49] Delta energy consumed for CPU with cpu_load : 0.000092 kWh, power : 9.707071602514286 W
[codecarbon INFO @ 10:07:49] Energy consumed for All CPU : 0.000092 kWh
[codecarbon INFO @ 10:07:49] 0.000188 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 5/5 - Loss: 1.1440
Energy: 0.00018752409770887867
Emissions: 3.2638569206230335e-05
Duration: 31.159603584999786


In [ ]:
from codecarbon import EmissionsTracker
import time


def run_client_experiment(
    client_id,
    client_loader
):

    model = SimpleCNN()

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    tracker = EmissionsTracker(
        save_to_file=False
    )

    tracker.start()

    tracker.start_task(
        f"client_{client_id}"
    )

    train_model(
        model,
        client_loader,
        optimizer,
        criterion,
        EPOCHS
    )

    client_data = tracker.stop_task()

    tracker.stop()

    result = {

    # ==================================================
    # Identificación
    # ==================================================

    "experiment_id": time.strftime("%Y-%m-%d_%H-%M-%S"),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),

    # ==================================================
    # Escenario experimental
    # ==================================================

    "scenario": SCENARIO,
    "record_type": "client",

    # ==================================================
    # Dataset y modelo
    # ==================================================

    "dataset": DATASET,
    "model": "SimpleCNN",

    # ==================================================
    # Configuración de entrenamiento
    # ==================================================

    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "random_seed": RANDOM_SEED,

    # ==================================================
    # Configuración federada
    # ==================================================

    "num_clients": NUM_CLIENTS,
    "client_id": client_id,
    "round": 1,

    # ==================================================
    # Resultados del modelo
    # ==================================================

    "accuracy": None,

  

    return result

In [8]:
client_1_result = run_client_experiment(
    client_id=1,
    client_loader=client_loaders[0]
)

print(client_1_result)

[codecarbon WARNING @ 10:13:32] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 10:13:32] >>> Tracker's metadata:
[codecarbon INFO @ 10:13:32]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 10:13:32]   Python version: 3.12.3
[codecarbon INFO @ 10:13:32]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 10:13:32]   Available RAM : 7.632 GB
[codecarbon INFO @ 10:13:32]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 10:13:32]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 10:13:32]   GPU count: 0
[codecarbon INFO @ 10:13:32]   GPU model: 
[codecarbon INFO @ 10:13:32] Energy consumed for RAM : 0.000000 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 10:13:32] Delta energy consumed for CPU with cpu_load : 0.000000 kWh, power : 4.8000432 W
[codecarbon INFO @ 10:13:32] Energy consumed for All CPU : 0.000000 kWh
[codecarbon INFO @ 10:13:32] 0.000000 kWh of electricity an

Epoch 1/5 - Loss: 1.9133
Epoch 2/5 - Loss: 1.5203
Epoch 3/5 - Loss: 1.3583
Epoch 4/5 - Loss: 1.2523


[codecarbon INFO @ 10:14:09] Energy consumed for RAM : 0.000094 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 10:14:09] Delta energy consumed for CPU with cpu_load : 0.000091 kWh, power : 9.73153287882353 W
[codecarbon INFO @ 10:14:09] Energy consumed for All CPU : 0.000091 kWh
[codecarbon INFO @ 10:14:09] 0.000185 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 5/5 - Loss: 1.1596
{'experiment_id': '2026-10-04_10-14-09', 'timestamp': '2026-10-04 10:14:09', 'scenario': 'federated', 'record_type': 'client', 'dataset': 'CIFAR10', 'model': 'SimpleCNN', 'epochs': 5, 'batch_size': 64, 'learning_rate': 0.001, 'random_seed': 42, 'num_clients': 5, 'client_id': 1, 'round': 1, 'energy_kwh': 0.0001845395068022784, 'cpu_energy_kwh': 9.101584719672096e-05, 'ram_energy_kwh': 9.352365960555744e-05, 'operational_emissions_kg': 3.2119101158936556e-05, 'duration_s': 30.576988130000245}


In [12]:
client_results = []

for i in range(NUM_CLIENTS):

    result = run_client_experiment(
        client_id=i + 1,
        client_loader=client_loaders[i]
    )

    client_results.append(result)

    print(
        f"Cliente {i+1} completado"
    )


[codecarbon WARNING @ 10:28:50] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 10:28:50] >>> Tracker's metadata:
[codecarbon INFO @ 10:28:50]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 10:28:50]   Python version: 3.12.3
[codecarbon INFO @ 10:28:50]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 10:28:50]   Available RAM : 7.632 GB
[codecarbon INFO @ 10:28:50]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 10:28:50]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 10:28:50]   GPU count: 0
[codecarbon INFO @ 10:28:50]   GPU model: 
[codecarbon INFO @ 10:28:50] Energy consumed for RAM : 0.000000 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 10:28:50] Delta energy consumed for CPU with cpu_load : 0.000000 kWh, power : 4.800014817600001 W
[codecarbon INFO @ 10:28:50] Energy consumed for All CPU : 0.000000 kWh
[codecarbon INFO @ 10:28:50] 0.000000 kWh of electr

Epoch 1/5 - Loss: 1.8763
Epoch 2/5 - Loss: 1.5063
Epoch 3/5 - Loss: 1.3519
Epoch 4/5 - Loss: 1.2455


[codecarbon INFO @ 10:29:27] Energy consumed for RAM : 0.000097 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 10:29:27] Delta energy consumed for CPU with cpu_load : 0.000095 kWh, power : 9.784388037257143 W
[codecarbon INFO @ 10:29:27] Energy consumed for All CPU : 0.000095 kWh
[codecarbon INFO @ 10:29:27] 0.000192 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 10:29:27] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 10:29:27] >>> Tracker's metadata:
[codecarbon INFO @ 10:29:27]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 10:29:27]   Python version: 3.12.3
[codecarbon INFO @ 10:29:27]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 10:29:27]   Available RAM : 7.632 GB
[codecarbon INFO @ 10:29:27]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 10:29:27]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 10:29:2

Epoch 5/5 - Loss: 1.1624
Cliente 1 completado
Epoch 1/5 - Loss: 1.8727
Epoch 2/5 - Loss: 1.5257
Epoch 3/5 - Loss: 1.3751
Epoch 4/5 - Loss: 1.2725


[codecarbon INFO @ 10:30:04] Energy consumed for RAM : 0.000096 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 10:30:04] Delta energy consumed for CPU with cpu_load : 0.000094 kWh, power : 9.766430643977145 W
[codecarbon INFO @ 10:30:04] Energy consumed for All CPU : 0.000094 kWh
[codecarbon INFO @ 10:30:04] 0.000190 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 10:30:04] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 10:30:04] >>> Tracker's metadata:
[codecarbon INFO @ 10:30:04]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 10:30:04]   Python version: 3.12.3
[codecarbon INFO @ 10:30:04]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 10:30:04]   Available RAM : 7.632 GB
[codecarbon INFO @ 10:30:04]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 10:30:04]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 10:30:0

...

Epoch 5/5 - Loss: 1.2431
Cliente 3 completado
Epoch 1/5 - Loss: 1.8473
Epoch 2/5 - Loss: 1.5117
Epoch 3/5 - Loss: 1.3826
Epoch 4/5 - Loss: 1.2542


[codecarbon INFO @ 10:31:20] Energy consumed for RAM : 0.000097 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 10:31:20] Delta energy consumed for CPU with cpu_load : 0.000096 kWh, power : 9.886753638994287 W
[codecarbon INFO @ 10:31:20] Energy consumed for All CPU : 0.000096 kWh
[codecarbon INFO @ 10:31:20] 0.000192 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 10:31:20] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 10:31:20] >>> Tracker's metadata:
[codecarbon INFO @ 10:31:20]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 10:31:20]   Python version: 3.12.3
[codecarbon INFO @ 10:31:20]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 10:31:20]   Available RAM : 7.632 GB
[codecarbon INFO @ 10:31:20]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 10:31:20]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 10:31:2

Epoch 5/5 - Loss: 1.1652
Cliente 4 completado
Epoch 1/5 - Loss: 1.8418
Epoch 2/5 - Loss: 1.4757
Epoch 3/5 - Loss: 1.3391
Epoch 4/5 - Loss: 1.2266


[codecarbon INFO @ 10:31:57] Energy consumed for RAM : 0.000095 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 10:31:57] Delta energy consumed for CPU with cpu_load : 0.000093 kWh, power : 9.76543103424 W
[codecarbon INFO @ 10:31:57] Energy consumed for All CPU : 0.000093 kWh
[codecarbon INFO @ 10:31:57] 0.000188 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 5/5 - Loss: 1.1139
Cliente 5 completado


In [13]:
import pandas as pd

clients_df = pd.DataFrame(client_results)

clients_df

,experiment_id,timestamp,scenario,record_type,dataset,model,epochs,batch_size,learning_rate,random_seed,num_clients,client_id,round,energy_kwh,cpu_energy_kwh,ram_energy_kwh,operational_emissions_kg,duration_s
0,2026-10-04_10-29-27,2026-10-04 10:29:27,federated,client,CIFAR10,SimpleCNN,5,64,0.001,42,5,1,1,0.000192,0.000095,0.000097,0.000033,31.770025
1,2026-10-04_10-30-04,2026-10-04 10:30:04,federated,client,CIFAR10,SimpleCNN,5,64,0.001,42,5,2,1,0.000190,0.000094,0.000096,0.000033,31.554369
2,2026-10-04_10-30-43,2026-10-04 10:30:43,federated,client,CIFAR10,SimpleCNN,5,64,0.001,42,5,3,1,0.000195,0.000096,0.000099,0.000034,32.481648
3,2026-10-04_10-31-20,2026-10-04 10:31:20,federated,client,CIFAR10,SimpleCNN,5,64,0.001,42,5,4,1,0.000192,0.000096,0.000097,0.000033,31.715864
4,2026-10-04_10-31-57,2026-10-04 10:31:57,federated,client,CIFAR10,SimpleCNN,5,64,0.001,42,5,5,1,0.000188,0.000093,0.000095,0.000033,31.111735


In [14]:
clients_df.to_csv(
    "../results/fl5_clients.csv",
    index=False
)

------------FIN DE LO NUEVO---------

In [ ]:
import time
from pathlib import Path
import pandas as pd

from codecarbon import EmissionsTracker

def run_experiment(
    scenario,
    model,
    train_loader,
    test_loader,
    optimizer,
    criterion,
    epochs
):

    tracker = EmissionsTracker(save_to_file=False)

    start_time = time.time()

    tracker.start()

    tracker.start_task("training")

    train_model(
        model,
        train_loader,
        optimizer,
        criterion,
        epochs
    )

    training_data = tracker.stop_task()

    emissions = tracker.stop()

    training_time = (
        time.time() - start_time
    )

    accuracy = evaluate_model(
        model,
        test_loader
    )

    print(f"Accuracy: {accuracy:.4f}")

    result = {
    
    # ==================================================
    # Identificación (anonimizada) de la máquina y del experimento
    # ==================================================

    "experiment_id": time.strftime("%Y-%m-%d_%H-%M-%S"),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),

    # ==================================================
    # Escenario experimental
    # ==================================================

    "scenario": scenario,
    "record_type": "experiment",

    # ==================================================
    # Dataset y modelo
    # ==================================================

    "dataset": DATASET,
    "model": "SimpleCNN",

    # ==================================================
    # Configuración de entrenamiento
    # ==================================================

    "epochs": epochs,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "random_seed": RANDOM_SEED,

    # ==================================================
    # Configuración federada
    # ==================================================

    "num_clients": None,
    "client_id": None,
    "round": None,

    # ==================================================
    # Resultados del modelo
    # ==================================================

    "accuracy": accuracy,

    # ==================================================
    # Tiempo
    # ==================================================

    "training_time_s": training_time,
    "duration_s": training_data.duration,

    # ==================================================
    # Energía
    # ==================================================

    "energy_kwh": training_data.energy_consumed,
    "cpu_energy_kwh": training_data.cpu_energy,
    "ram_energy_kwh": training_data.ram_energy,

    # ==================================================
    # Emisiones
    # ==================================================

    "operational_emissions_kg": training_data.emissions,
    "carbon_intensity": None,

        
    # ==================================================
    # SCI (Software Carbon Intensity)
    # ==================================================
    
    "embedded_carbon_kg": None,
    "sci": None,
}

    results_file = "../results/experiments.csv"

    df = pd.DataFrame([result])

    if Path(results_file).exists():

        df.to_csv(
            results_file,
            mode="a",
            header=False,
            index=False
        )

    else:

        df.to_csv(
            results_file,
            index=False
        )

    return result

In [ ]:
accuracy = evaluate_model(
    model,
    test_loader
)

print(accuracy)

Ejecución del experimento:

In [ ]:
import pandas as pd
from pathlib import Path
result = run_experiment(
    scenario=SCENARIO,
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS
)

print(result)

In [ ]:

import pandas as pd
from pathlib import Path

results = pd.read_csv(
    "../results/experiments.csv"
)

results